# Statistical analysis of the data

This notebook consists of statistical checks of the inspected variables, related to the hypothesis set for this part of analysis: 

RQ: How do selected manifestations of political identity (political parties, party status, political orientation) shape parliamentary discourse in terms of sentiment?

- H1. Coalition parties will consistently exhibit higher neutral/positive sentiment, while opposition parties will display more negativity, though intensity may vary across terms based on political events.
- H2: Changes in political orientation will correlate with sentiment shifts, with right/traditionalist parties showing higher negativity.
- H3: Negative sentiment will dominate parliamentary debates, with minor occurrences of other sentiment categories. Political crises, elections, or leadership changes will correlate with sentiment shifts.

The sample illustrates the workflow. The full analysis is run on the complete dataset for final results.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats


In [2]:
df = pd.read_csv("../Results/Datasets/ParlaMint-SI_Family.tsv", sep="\t", encoding="utf-8")
df.head()
df.shape

(3534, 26)

## H1: Coalition parties more neutral/positive; opposition more negative

Given the non-normal distributions of sentiment checks, we will use Mann–Whitney U test to check several options: 

- a) is there a (statistically significant) difference between the two variables?
    - Null hypothesis (H₀): The distribution of sentiment scores is the same in coalition and opposition parties.

For this we will first run two-sided test (to compare if there is a difference between coalition and opposition in terms of sentiment).

- b) if the results are statistically significant, we will check the direction with the one-sided test
- c) Since procedural speeches are, as seen in distribution check, mostly neutral in sentiment (Neutral - 94.56%, Negative - 4.53%, Positive - 0.89%),
 but the actual chairs are usually directly affiliated with political parties/parliamentary groups, we will run the test: 
    - first on raw sentiment scores, separated by coalition and opposition (keeping the dataset intact, inspecting full picture)
    - and for a second run, we will remove the procedural speeches from the dataset as a robustnes check and to see if results from first run still hold

### Two-sided Mann–Whitney U (entire dataset)

In [ ]:
from scipy.stats import mannwhitneyu

coal_scores = df[df["Party_status"] == "Coalition"]["Senti_n"]
oppo_scores = df[df["Party_status"] == "Opposition"]["Senti_n"]

u_stat, p_two = mannwhitneyu(coal_scores, oppo_scores, alternative="two-sided")
print("Mann–Whitney U statistic:", u_stat)
print("Two-sided p-value:", p_two)

n1, n2 = len(coal_scores), len(oppo_scores)
effect_size_r = 1 - (2*u_stat) / (n1*n2)
print("Effect size (r):", effect_size_r)
print("Coalition median:", coal_scores.median())
print("Opposition median", oppo_scores.median())


Mann–Whitney U statistic: 1645543.0
Two-sided p-value: 3.046903445819607e-70
Effect size (r): -0.36230653041473726
Coalition median: 3.0
Opposition median 1.18


Sentiment scores differed significantly between coalition and opposition parties (Mann–Whitney U = 1,645,543, p < 0.001). The median sentiment was higher for coalition parties, consistent with the hypothesis that coalition MPs speak in a more neutral/positive tone compared to opposition MPs. 

Given the statistically significant results in the two-sided test, we then test for direction with a one-sided test.

### One-sided Mann-Whitney U test (entire dataset)


In [ ]:
from itertools import product
u_stat, p_one = mannwhitneyu(coal_scores, oppo_scores, alternative='greater')

print("U:", u_stat, "One-sided p-value (coalition > opposition):", p_one)

# Rank-biserial effect size --> Count how many coalition scores are higher than opposition scores
favorable = sum(c > o for c, o in product(coal_scores, oppo_scores))
unfavorable = sum(c < o for c, o in product(coal_scores, oppo_scores))
r = (favorable - unfavorable) / (n1 * n2)
print("Effect size (rank-biserial r):", r)

U: 1645543.0 One-sided p-value (coalition > opposition): 1.5234517229098035e-70
Effect size (rank-biserial r): 0.3623065304147372


The one-sided test cofirms that the direction is statistically significant (coalition > opposition). The more robust interpetation of the Rank-biserial effect size also points out that the effect size (r=0.36) is medium, signifying a moderate difference in sentiment distribution between coalition and opposition.

The test is repeated, this time removing speeches made by chairspeaker

In [5]:
df_reg = df[df["Speaker_role"] == "Regular"]
df_reg.shape

(1748, 26)

### Two-sided Mann-Whitney U test, removed procedural speech

For the second run, we remove procedural speeches which can increase the number of neutral sentiment speeches and reduce the contrast between coalition and opposition sentiment.

In [6]:
#Two-sided Mann-Whitney U test, removed procedural speech

coal = df_reg[df_reg["Party_status"] == "Coalition"]["Senti_n"]
oppo = df_reg[df_reg["Party_status"] == "Opposition"]["Senti_n"]

u_stat, p_two = mannwhitneyu(coal, oppo, alternative="two-sided")
print("Mann–Whitney U statistic:", u_stat)
print("Two-sided p-value:", p_two)

favorable = sum(c > o for c, o in product(coal, oppo))
unfavorable = sum(c < o for c, o in product(coal, oppo))
r = (favorable - unfavorable) / (n1 * n2)
print("Effect size (rank-biserial r):", r)

print("Coalition median:", coal.median())
print("Opposition median", oppo.median())


Mann–Whitney U statistic: 380540.5
Two-sided p-value: 2.711033572845041e-54
Effect size (rank-biserial r): 0.10152457613753348
Coalition median: 1.435
Opposition median 0.19


- The p-value shows highly significant difference between distributions, even after excluding procedural speeches.
- The effect size r = 0.10 indicates the difference is statistically reliable but small in magnitude.
- The medians suggest coalition still has higher sentiment than opposition, consistent with your hypothesis — but the difference is less pronounced when procedural content is excluded.

### The two-sided test with removed procedural speech

In [ ]:
u_stat, p_one = mannwhitneyu(coal, oppo, alternative="greater")
print("Mann–Whitney U statistic:", u_stat)
print("One-sided p-value:", p_one)

Mann–Whitney U statistic: 380540.5
Two-sided p-value: 1.3555167864225205e-54


The one-sided test cofirms that the direction is statistically significant (coalition > opposition), even after excluding procedural speeches. 